# QLoRA runner for the red-team attacker model
Run on a free Colab or Kaggle GPU. This notebook never runs on the local machine. It installs training dependencies, trains on the processed dataset, saves checkpoints, evaluates the base versus tuned model, and serves the adapter through a cloudflared tunnel.
The repo URL is set by default. Override it with the `REPO_URL` environment variable if you fork. No Hugging Face account is needed. Qwen2.5-3B-Instruct is a public model.
On Kaggle: set Notebook options, Accelerator to GPU T4 x2, and Internet to On. Then Run All.

In [ ]:
import os
from pathlib import Path
BASE = Path('/kaggle/working') if Path('/kaggle').exists() else Path('/content')
os.chdir(BASE)
REPO_URL = os.environ.get('REPO_URL', 'https://github.com/DSCmatter/Fencio_ass.git')
if os.path.isdir('redteam-agent'):
    !git -C redteam-agent fetch origin main
    !git -C redteam-agent reset --hard origin/main
else:
    !git clone -q {REPO_URL} redteam-agent
%cd redteam-agent
!pwd
!git log --oneline -1

In [ ]:
# pip 26.3 blocks imports during build isolation. Pin below it.
from pathlib import Path
!python -m pip install -q "pip<26.3"
if Path('/kaggle').exists():
    !python -m pip install -q unsloth trl transformers datasets accelerate peft bitsandbytes fastapi uvicorn pyyaml httpx
else:
    !python -m pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" trl transformers datasets accelerate peft bitsandbytes fastapi uvicorn pyyaml httpx
    !python -m pip install -q --no-deps xformers

In [ ]:
# Sync to latest main before training
!git fetch origin main
!git reset --hard origin/main
!grep base_model configs/train.yaml
!git log --oneline -1

In [ ]:
!python training/train_qlora.py --config configs/train.yaml --train-file data/processed/train.jsonl --output checkpoints/qlora-adapter

In [ ]:
import shutil
from pathlib import Path
drive = Path('/content/drive/MyDrive/redteam-checkpoints')
if Path('/content/drive').exists():
    drive.mkdir(parents=True, exist_ok=True)
    shutil.copytree('checkpoints/qlora-adapter', drive / 'qlora-adapter', dirs_exist_ok=True)
    print('checkpoint copied to', drive)
else:
    print('checkpoint stays at checkpoints/qlora-adapter')

In [ ]:
!python training/evaluate_model.py --config configs/train.yaml --test-file data/processed/test.jsonl --adapter checkpoints/qlora-adapter --output eval/results/model_eval.json

## Remote inference
Run the next cell only when the assessment needs the fine-tuned attacker. It prints a cloudflared URL. Paste that URL into `configs/agent.yaml` as `model.base_url`, or export `RT_MODEL_BASE_URL` before running the local agent.

In [ ]:
!python training/serve_notebook.py --config configs/train.yaml --adapter checkpoints/qlora-adapter --port 8000